In [ ]:
# Setup: in Google Colab, download and install the repository (a few minutes, once per session); elsewhere, work from the
# repository. Pages that read the derived catalogues need Zenodo bundle 1 (python scripts/fetch_release.py).
import os, subprocess, sys
os.environ['MPLBACKEND'] = 'Agg'   # as in a plain script run: Jupyter's inline settings would change the saved figures
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/lmc-ancient-disc'):
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/vasilybelokurov/lmc-ancient-disc', '/content/lmc-ancient-disc'], check=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '/content/lmc-ancient-disc'], check=True)
    os.chdir('/content/lmc-ancient-disc')
else:
    while not os.path.exists('pyproject.toml'):
        os.chdir('..')
sys.path[:0] = ['src', 'scripts', 'kb']
if not os.path.exists('data/giants_apogee.parquet'):   # the catalogues of Zenodo bundle 1, checked against data/sources.yaml
    subprocess.run([sys.executable, 'scripts/fetch_release.py'], check=True)

# Metallicity structure

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/lmc-ancient-disc/blob/main/kb/notebooks/chapters/metallicity.ipynb)

Every number on this page is written by `scripts/paper_text_feh.py` from the result tables (`results/macros_feh.tex`); the code
cells are the figure scripts themselves (`scripts/paper_feh_profile.py`, `paper_feh_literature.py`, `paper_mdf_literature.py`), step
by step.

Metallicity here is the iron abundance [Fe/H], the logarithm of the ratio of iron to hydrogen atoms relative to the Sun's. Three
tracers measure it: red giants with spectra from [APOGEE DR17](https://doi.org/10.3847/1538-4365/ac4414) (near-infrared spectra
analysed by the ASPCAP pipeline) and from the BOSS optical spectra of [SDSS-V DR20](https://doi.org/10.48550/arXiv.2607.26149)
(analysed by the neural network BOSSNet, [Sizemore et al. 2024](https://doi.org/10.3847/1538-3881/ad291d), and calibrated to the
APOGEE scale; chapter 3), and RR Lyrae stars of type ab (RRab, pulsating old stars) from the
[OGLE-IV](https://ui.adsabs.harvard.edu/abs/2015AcA....65....1U) and [*Gaia* DR3](https://doi.org/10.1051/0004-6361/202243940) surveys,
whose [Fe/H] is photometric, estimated from the shape of their light curves on the scale of
[Crestani et al. (2021)](https://doi.org/10.3847/1538-4357/abd183) (chapter 3). $G$ is the *Gaia* magnitude.

With the selection functions of chapter 2 the giant samples give the metallicity distribution of the LMC at each radius, not that of
the fields the surveys chose to observe. We place every star on the disc plane of
[van der Marel & Kallivayalil (2014)](https://doi.org/10.1088/0004-637X/781/2/121) (inclination 34°, line of nodes at
position angle 139.1°, distance of the centre $D_0$ = 50.1 kpc; with the inclination of our joint
kinematic fit, 28° (chapter 6), radii on the minor axis would be 6 per cent smaller and those
on the line of nodes unchanged) and use its in-plane distance $R$ from the centre, in 1-kpc bins to
$R$ = 10 kpc, the largest radius at which the 12° BOSS footprint covers all azimuths. Each BOSS giant
carries its selection weight $w_i$ (the number of stars of the BOSS target list that it stands for), its coverage factor $c_i$ (which
adds the target-list stars in sky areas with too few spectra) and one minus its Milky Way foreground fraction $f_{\rm MW}$ (all from
chapters 2 and 4); each APOGEE giant carries its selection weight. The RR Lyrae are unweighted, since the union of the two surveys is
at least 96 per cent complete at every radius (chapter 2). In each bin we take the weighted median of the measured [Fe/H]. For the BOSS giants we also
count the stars in three ranges of true metallicity, below -1.5, from -1.5 to -1 and
above -1, and divide by the area of the ring on the disc plane. These counts must be corrected for the BOSS
metallicity errors, which move stars from −2 < [Fe/H] < −1.5 to below −2 (chapter 4). The correction applies to counts, not to single
stars: in each radial bin we rebuild the response matrix (the fraction of stars of each true [Fe/H] bin that BOSS places in each
measured bin) for the $G$ and signal-to-noise distribution of the bin's giants and fit the response equation of chapter 4, with the
efficiencies set to 1, to their weighted counts in 0.125-dex measured bins, which returns the counts in the true bins.
The medians are of the measured values; above [Fe/H] = −1.5 calibrated BOSS agrees with APOGEE to within 0.03 dex
in median (chapter 4). Errors are the 16th–84th percentiles of 200 bootstrap draws of HEALPix
([Górski et al. 2005](https://doi.org/10.1086/427976)) sky pixels at resolution nside = 64 (0.9°), with
the weights fixed and the 1 923 stars that define the response resampled in each draw (8 of the
2 000 bin fits, where a resampled response had an empty column, are dropped); the coarser pixels of the mass bootstrap
would leave one or two pixels in the inner bins.

<details><summary>About <code>scripts/paper_feh_profile.py</code></summary>

```text
Metallicity against in-plane radius for the BOSS and APOGEE giants and the RRab.

Radius R [kpc]: distance from the centre in the adopted disc plane (lmcold.constants.LMC: inclination, line of nodes, D0), each
star placed on the plane. The 12-deg BOSS footprint covers all azimuths only to R ~ 10 kpc, so bins stop there.
Giants (lmcold.mass_samples.samples) are counted with their selection weights; BOSS also with the coverage factor and one minus the
Milky Way foreground fraction (data/giant_foreground.parquet, from build_giant_mass.py). APOGEE covers only its 35 fields.
RRab members are unweighted: the Gaia+OGLE union is >= 96 per cent complete at all radii (Sect. 2.2.1).
(a, b) weighted median [Fe/H] of the giants (BOSS also with G < G_BRIGHT) and the RRab, with the published gradients (lmcold.constants.
LMC_FEH_GRADIENT_LITERATURE) drawn over their own radial ranges through our profile at the middle of each range (slopes only);
(c) surface density of the BOSS giants [kpc^-2 on the disc plane] in three ranges of true [Fe/H]: per R bin, the weighted counts
in fine measured bins are fitted through the BOSS response (lmcold.mass_samples.boss_response re-weighted to the bin's G mix,
lmcold.mass_response.recover_mass), so no star is corrected individually; the medians in (a) are of measured [Fe/H]. Dashed:
exponentials fitted to ln(Sigma) over FIT_RANGE; scale lengths in results/feh_profile.json with their bootstrap percentiles and
the foreground set to zero and doubled. Percentiles 16-84 of [Fe/H] are in the csv. Open symbols: RRab members per kpc^2 below and
above [Fe/H] = RR_SPLIT (photometric, Crestani+21 scale; unweighted, Poisson errors), to RR_R_MAX, with exponentials over
RR_FIT_RANGE; their scale lengths are in results/feh_profile.json.
Errors: 16th-84th percentiles of NBOOT draws of nside-64 (0.9-deg) sky pixels with the weights fixed and, for (c), the overlap
stars of the response resampled; nside 16 would leave one or two pixels in the inner bins.
Writes figures/paper_feh_profile.png, results/feh_profile.csv, results/feh_profile.json and, against projected distance rho [deg] for
scripts/paper_feh_literature.py, results/feh_profile_rho.csv.

Inputs:
- lmcold.mass_samples.samples(): BOSS and APOGEE mass samples (velocity-selected giants with selection weights w; BOSS [Fe/H] on
  the APOGEE scale) from data/boss_weights.parquet (sources.yaml release_boss_weights), data/apogee_weights.parquet
  (release_apogee_weights), data/spec_giants.parquet and results/boss_calibration.json.
- data/giant_foreground.parquet (scripts/build_giant_mass.py): per BOSS giant `fill` (coverage factor) and `f_mw`.
- data/rrab.parquet (release_rrab, scripts/build_rrab.py): RRab with `member` flag and adopted photometric `feh`.
Outputs:
- results/feh_profile.csv (bins of R [kpc]) and results/feh_profile_rho.csv (bins of rho [deg]): one row per sample (BOSS,
  APOGEE, RRab, 'BOSS G<16.5') and bin, with columns sample, R_lo/R_hi (or rho_lo/rho_hi), n (stars), median, p16, p84
  (weighted percentiles of [Fe/H] in the bin), median_lo/median_hi (16th/84th percentiles of the median over the bootstrap);
  BOSS in R also sigma_<range>, sigma_<range>_lo, sigma_<range>_hi [giants kpc^-2] for range lt15, 15to10, gt10.
- results/feh_profile.json: per giant range h [kpc], h_p16, h_p84 (bootstrap), h_foreground_x0, h_foreground_x2; fit range, NBOOT,
  number of skipped bootstrap bins; per RRab range (rr_lt15, rr_gt15) h [kpc], fit range and counts per R bin.
Panels (1 x 3; legends above the panels; x: in-plane radius R [kpc], 0-10):
(a) y: weighted median [Fe/H] [dex] of BOSS (solid), BOSS G < 16.5 (dashed, open), APOGEE, error bars the bootstrap 16-84 range
    of the median; published giant gradients as line segments over their radial ranges (slopes only).
(b) the same for the RRab with the Skowron+16 inner, middle and outer segments.
(c) y: number per kpc^2 on the disc plane (log); filled circles: BOSS giants in true [Fe/H] < -1.5, -1.5 to -1, > -1 with
    exponential fits over FIT_RANGE (dashed); open squares: RRab below and above RR_SPLIT with exponential fits over RR_FIT_RANGE
    (dotted). Legend gives the scale lengths h.
Figure: paper Fig. 5 (Sect. 3, main text; docs/figures.md).
Run: .venv/bin/python scripts/paper_feh_profile.py
Run time: 11 s on a MacBook (M-series), docs/run_times.txt.
```
</details>

**Data read by `scripts/paper_feh_profile.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `data/apogee_weights.parquet` | Zenodo bundle 1 | APOGEE selection weights per candidate star (selection groups, coverage, w = N_cand / N_quality x pass factor in H x XP [M/H]) |
| `data/boss_weights.parquet` | Zenodo bundle 1 | BOSS selection weights per eligible parent star (selection groups, w = N_elig / N_good x pass factor in G x XP [M/H] x ring) |
| `data/giant_foreground.parquet` | Zenodo bundle 1 | Milky Way foreground model per giant, from the mass analysis |
| `data/rrab.parquet` | Zenodo bundle 1 | RRab within 20 deg: OGLE-IV + Gaia DR3, [Fe/H], dmu, membership |
| `data/spec_giants.parquet` | Zenodo bundle 1 | APOGEE DR17 and SDSS-V DR20 BOSS spectra of LMC giant candidates before the Gaia match |
| `results/boss_calibration.json` | repository | calibration of the BOSS [Fe/H] and velocities to APOGEE on the stars observed by both; written by `python scripts/build_giants.py` |

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import json
import healpy as hp
import numpy as np
import pandas as pd
from lmcold.constants import LMC, LMC_FEH_GRADIENT_LITERATURE
from lmcold.footprint import angular_distance
from lmcold.layout import panel_grid
from lmcold.maps import wquantile
from lmcold.mass_response import recover_mass
from lmcold.mass_samples import samples, overlap, boss_response, MEASURED_EDGES, TRUE_EDGES
from lmcold.perspective import disc_radius
from lmcold.sources import DATA, ROOT

## Settings and helpers
Radial bins, the true [Fe/H] ranges and fit ranges, and the helpers for the profiles, the scale lengths and the bootstrap.

In [ ]:
R_EDGES = np.arange(0., 10.5, 1.)  # kpc, 1-kpc in-plane radius bins 0-10 kpc
# BOSS reaches G = 17.5; 93 per cent of the APOGEE weight is at G < 16.5 (its faint groups are uncovered), and the two medians agree
# at G < 16.5 (scripts/fig_feh_profile_glim.py), so the BOSS median is also shown to this limit.
G_BRIGHT = 16.5
RHO_EDGES = np.arange(0., 12.5, 1.)  # projected distance [deg], for literature quoted in degrees
NBOOT = 200  # bootstrap draws of sky pixels
R_MID = .5 * (R_EDGES[:-1] + R_EDGES[1:])
# True [Fe/H] ranges for the surface densities: indices of the TRUE_EDGES bins (-3, -2, -1.5, -1, 0.5) they sum.
RANGES = {'lt15': (0, 1), '15to10': (2,), 'gt10': (3,)}
FIT_RANGE = (2., 10.)  # kpc; inside 2 kpc the bar adds to the metal-rich profile
# RRab members lie within rho < 10 deg, which covers all azimuths of the disc plane to R = 8 kpc (93 per cent of 8-9 kpc, 32 per cent
# of 9-10 kpc; points uniform on the sky), so their profiles stop there.
RR_SPLIT, RR_R_MAX = -1.5, 8.  # RRab [Fe/H] split [dex] and outer radius of the RRab profiles [kpc]
RR_FIT_RANGE = (2., RR_R_MAX)
rng = np.random.default_rng(1)  # fixed seed: the bootstrap is reproducible


def profile(s, o=None, col='R', edges=R_EDGES):
    """Per bin of s[col]: median, p16, p84 of [Fe/H]; with the overlap stars o (BOSS), also the surface density [giants kpc^-2]
    in the RANGES of true [Fe/H], from the counts in fine measured bins fitted through the response (Sect. 2.3).

    Parameters
    ----------
    s : pandas.DataFrame
        Sample with columns feh [dex], W (weight per star), col, and for the surface densities g (Gaia G) and snr (BOSS S/N).
    o : pandas.DataFrame or None, optional
        Overlap stars (lmcold.mass_samples.overlap) defining the BOSS response; None skips the surface densities.
    col : str, optional
        Binned column: 'R' (in-plane radius [kpc]) or 'rho' (projected distance [deg]).
    edges : numpy.ndarray, optional
        Bin edges in `col`.

    Returns
    -------
    numpy.ndarray
        Shape (len(edges) - 1, 3 + len(RANGES)): weighted median, 16th and 84th percentiles of [Fe/H] [dex], then the surface
        density per RANGES entry [weighted giants kpc^-2, annulus area pi (R_hi^2 - R_lo^2) on the disc plane]. NaN for bins with
        total weight <= 0 or fewer than 20 stars, and for densities where the response cannot be built.
    """
    k = np.digitize(s[col], edges) - 1
    out = np.full((len(edges) - 1, 3 + len(RANGES)), np.nan)
    for i in range(len(edges) - 1):
        t = s[k == i]
        if t.W.sum() <= 0 or len(t) < 20:
            continue
        x, w = t.feh.to_numpy(), t.W.to_numpy()
        out[i, :3] = wquantile(x, w, [.5, .16, .84])
        if o is not None and (x < -1.5).any():  # boss_response needs stars on both sides of -1.5
            resp = boss_response(o, x, t.g.to_numpy(), t.snr.to_numpy(), w)
            if not np.isfinite(resp).all():  # a resampled overlap can leave a column empty after G reweighting; draw skipped
                continue
            counts = np.histogram(x, MEASURED_EDGES, weights=w)[0]  # weighted counts in 0.125-dex measured [Fe/H] bins
            true = recover_mass(counts, resp, np.ones(len(TRUE_EDGES) - 1))  # eta = 1: returns counts
            area = np.pi * (edges[i + 1] ** 2 - edges[i] ** 2)
            out[i, 3:] = [true[list(j)].sum() / area for j in RANGES.values()]
    return out


def scale_lengths(v):
    """Exponential scale length [kpc] of each RANGES profile, from a straight line in ln(Sigma) over FIT_RANGE.

    Unweighted least squares on the bin centres R_MID strictly inside FIT_RANGE; Sigma(R) ~ exp(-R / h), so h = -1 / slope.

    Parameters
    ----------
    v : numpy.ndarray
        Output of `profile` in R_EDGES bins, shape (len(R_MID), 3 + len(RANGES)).

    Returns
    -------
    list of float
        Scale length h [kpc] per RANGES entry.
    """
    m = (R_MID > FIT_RANGE[0]) & (R_MID < FIT_RANGE[1])
    return [-1 / np.polyfit(R_MID[m], np.log(v[m, 3 + j]), 1)[0] for j in range(len(RANGES))]


def boot(s, o=None, col='R', edges=R_EDGES):
    """NBOOT draws of nside-64 sky pixels; the overlap stars that define the response are resampled in each draw.

    Each draw picks len(upix) pixels with replacement among the occupied ones and multiplies the weight of every star by the
    number of times its pixel was drawn (the selection weights themselves are not recomputed).

    Parameters
    ----------
    s : pandas.DataFrame
        Sample as for `profile`, plus `pix` (HEALPix nside-64 index, ring ordering).
    o, col, edges
        As for `profile`.

    Returns
    -------
    numpy.ndarray
        Shape (NBOOT, len(edges) - 1, 3 + len(RANGES)): `profile` of each draw.
    """
    pix = s.pix.to_numpy()
    upix, inv = np.unique(pix, return_inverse=True)
    draws = []
    for _ in range(NBOOT):
        m = np.bincount(rng.integers(0, len(upix), len(upix)), minlength=len(upix))[inv]
        t = s.assign(W=s.W * m)
        draws.append(profile(t[t.W > 0], None if o is None else o.sample(len(o), replace=True, random_state=rng), col, edges))
    return np.array(draws)

## Load the data
The BOSS and APOGEE mass samples through `lmcold.mass_samples.samples()` (built from the catalogues of Zenodo bundle 1), the
coverage fill and foreground fraction of each BOSS giant (`data/giant_foreground.parquet`, written by
`scripts/build_giant_mass.py`), and the RRab catalogue `data/rrab.parquet` (Zenodo bundle 1).

In [ ]:
# lmcold.mass_samples.samples(): BOSS and APOGEE giants with calibrated [Fe/H], g, snr and selection weight w
b, a, _ = samples()
# data/giant_foreground.parquet: per BOSS giant the coverage fill and the Milky Way foreground fraction f_mw
fg = pd.read_parquet(DATA / 'giant_foreground.parquet')
# data/rrab.parquet: RRab with member flag and photometric [Fe/H]
r = pd.read_parquet(DATA / 'rrab.parquet')

## Samples and weights

In [ ]:
# --- Samples and weights ---
o = overlap(b, a)  # stars with both spectra: feh_b (calibrated BOSS), feh_a (APOGEE), BOSS g and snr
b = b.merge(fg[['source_id', 'fill', 'f_mw']], on='source_id', validate='one_to_one')
b['W'] = b.w * b['fill'] * (1 - b.f_mw)  # BOSS: selection weight x coverage fill x (1 - Milky Way foreground fraction)
a['W'] = a.w  # APOGEE: selection weight only (its 35 fields)
r = r[r.member & np.isfinite(r.feh)].assign(W=1.)
for s in (b, a, r):
    # R: in-plane radius [kpc] with each star placed on the adopted disc plane; rho: projected distance [deg] from (ra0, dec0).
    s['R'] = disc_radius(s.ra.to_numpy(), s.dec.to_numpy())
    s['rho'] = angular_distance(s.ra.to_numpy(), s.dec.to_numpy(), LMC['ra0'], LMC['dec0'])
    s['pix'] = hp.ang2pix(64, s.ra.to_numpy(), s.dec.to_numpy(), lonlat=True)

## Profiles and bootstrap errors
Against projected distance and in-plane radius; written to `results/feh_profile_rho.csv` and `results/feh_profile.csv`.

In [ ]:
# --- Profiles and bootstrap errors, against rho (for the appendix) and R (this figure; `t` and `summary` keep the R results) ---
for col, edges, out in [('rho', RHO_EDGES, 'feh_profile_rho.csv'), ('R', R_EDGES, 'feh_profile.csv')]:
    rows = []
    extra = [(f'BOSS G<{G_BRIGHT}', b[b.g < G_BRIGHT])]  # last, so the other bootstraps keep their draws
    for name, s in [('BOSS', b), ('APOGEE', a), ('RRab', r)] + extra:
        oo = o if (name, col) == ('BOSS', 'R') else None  # surface densities only for the full BOSS sample against R
        state = rng.bit_generator.state
        v, d = profile(s, oo, col, edges), boot(s, oo, col, edges)
        if col == 'rho' and name.endswith(str(G_BRIGHT)):  # added later: restore the generator so the R bootstraps keep their draws
            rng.bit_generator.state = state
        e = np.nanpercentile(d, [16, 84], axis=0)
        n = np.bincount(np.digitize(s[col], edges) - 1, minlength=len(edges))[:len(edges) - 1]
        for i in range(len(edges) - 1):
            row = {'sample': name, col + '_lo': edges[i], col + '_hi': edges[i + 1], 'n': n[i], 'median': v[i, 0], 'p16': v[i, 1],
                   'p84': v[i, 2], 'median_lo': e[0, i, 0], 'median_hi': e[1, i, 0]}
            if oo is not None:
                for j, key in enumerate(RANGES):
                    row.update({f'sigma_{key}': v[i, 3 + j], f'sigma_{key}_lo': e[0, i, 3 + j], f'sigma_{key}_hi': e[1, i, 3 + j]})
            rows.append(row)
        if oo is not None:
            h = np.array([scale_lengths(x) for x in d])
            summary = {'fit_range_kpc': FIT_RANGE, 'nboot': NBOOT, 'skipped_bins': int(np.isnan(d[:, :, 3]).sum())}
            for j, key in enumerate(RANGES):
                summary[key] = dict(h=scale_lengths(v)[j], h_p16=np.nanpercentile(h[:, j], 16), h_p84=np.nanpercentile(h[:, j], 84))
            for k_fg in (0., 2.):  # foreground bracket, as for the masses
                vf = profile(s.assign(W=s.w * s['fill'] * np.clip(1 - k_fg * s.f_mw, 0, None)), o, col, edges)
                for j, key in enumerate(RANGES):
                    summary[key][f'h_foreground_x{k_fg:g}'] = scale_lengths(vf)[j]
    t = pd.DataFrame(rows)
    t.to_csv(ROOT / 'results' / out, index=False)

## RR Lyrae surface densities
Counts per ring below and above the [Fe/H] split, with their scale lengths; written to `results/feh_profile.json`.

In [ ]:
# --- RRab surface density below and above RR_SPLIT, 0-8 kpc: members per kpc^2 on the disc plane ---
rr_bins = R_EDGES[R_EDGES <= RR_R_MAX]
rr_mid = .5 * (rr_bins[1:] + rr_bins[:-1])
rr_n = {key: np.histogram(r.R[m], rr_bins)[0] for key, m in [('rr_lt15', r.feh < RR_SPLIT), ('rr_gt15', r.feh >= RR_SPLIT)]}
rr_area = np.pi * np.diff(rr_bins ** 2)
fit = (rr_mid > RR_FIT_RANGE[0]) & (rr_mid < RR_FIT_RANGE[1])
for key, n in rr_n.items():
    # ln(N / area) has error 1 / sqrt(N) for Poisson counts, so polyfit weights (1 / sigma) are sqrt(N).
    cf = np.polyfit(rr_mid[fit], np.log(n[fit] / rr_area[fit]), 1, w=np.sqrt(n[fit]))
    summary[key] = dict(h=-1 / cf[0], fit_range_kpc=RR_FIT_RANGE, n=n.tolist())
(ROOT / 'results/feh_profile.json').write_text(json.dumps(summary, indent=1))
print(t.round(3).to_string(index=False))
print(json.dumps(summary, indent=1))

## The figure

In [ ]:
# --- Figure ---
plt.style.use(ROOT / 'src/lmcold/paper.mplstyle')
colours = dict(BOSS='#2878b5', APOGEE='#d75a20', RRab='#3a923a')
# Horizontal offsets [kpc] of each sample's points, so their error bars do not overlap.
shift = {'BOSS': 0., f'BOSS G<{G_BRIGHT}': .06, 'APOGEE': .12, 'RRab': -.12}
# Published gradients: one colour and dash pattern per study.
LIT_STYLE = {'Povick+25': ('#000000', (0, (6, 2))), 'Grady+21': ('#7b3294', (0, (1, 1.2))), 'Cioni 09': ('#a6611a', (0, (5, 1.5, 1, 1.5))),
             'Choudhury+16': ('#d01c8b', (0, (2.5, 1))), 'Choudhury+21': ('#4d9221', (0, (8, 1.5, 1, 1.5, 1, 1.5))),
             'Skowron+16': ('#000000', (0, (6, 2)))}
fig, axes = panel_grid(1, 3, width=7, left=.55, right=.1, hgap=.62, bottom=.5, top=.95)
for ax, samples_, tracer in [(axes[0, 0], ['BOSS', f'BOSS G<{G_BRIGHT}', 'APOGEE'], 'giants'), (axes[0, 1], ['RRab'], 'RRab')]:
    for name in samples_:
        q = t[t['sample'] == name]
        x = .5 * (q.R_lo + q.R_hi) + shift[name]
        bright = name.endswith(str(G_BRIGHT))
        ax.errorbar(x, q['median'], [q['median'] - q.median_lo, q.median_hi - q['median']], fmt='--o' if bright else '-o', ms=3, lw=1,
                    color=colours[name.split()[0]], mfc='white' if bright else None,
                    label=f'BOSS G < {G_BRIGHT} median' if bright else f'{name} median', zorder=3)
    # Published gradients, each over its own range and through our profile (BOSS for giants) at the middle of that range:
    # only the slopes are compared, so no zero points are needed.
    ref = t[t['sample'] == samples_[0]]
    for L in (x for x in LMC_FEH_GRADIENT_LITERATURE if x['tracer'] == tracer):
        hi = min(L['r_hi'], R_EDGES[-1])
        mid = .5 * (L['r_lo'] + hi)
        slope = L['slope'] if 'slope_hi' not in L else .5 * (L['slope'] + L['slope_hi'])
        y0 = np.interp(mid, .5 * (ref.R_lo + ref.R_hi), ref['median'])
        rr = np.array([L['r_lo'], hi])
        study = L['label'].split(' inner')[0].split(' middle')[0].split(' outer')[0]
        c, ls = LIT_STYLE[study]
        yy = y0 + slope * (rr - mid)
        ax.plot(rr, yy, color=c, lw=1.4, ls=ls, zorder=2, label=None if L['label'].endswith((' middle', ' outer')) else study)
    ax.set(xlabel='R [kpc]', ylabel='median [Fe/H]', xlim=(0, 10))
    ax.legend(loc='lower center', bbox_to_anchor=(.5, 1.02), ncol=2, frameon=False, fontsize=6.5, handlelength=3.5,
              columnspacing=1.)
axes[0, 0].set_title('(a) Giants', pad=62)
axes[0, 1].set_title('(b) RR Lyrae', pad=62)
# (c) Surface densities: giants by true [Fe/H] (response-corrected), RRab by photometric [Fe/H].
ax = axes[0, 2]
q = t[t['sample'] == 'BOSS']
for key, label, c in [('lt15', '[Fe/H] < −1.5', '#3b2c85'), ('15to10', '−1.5 < [Fe/H] < −1', '#2a9d8f'), ('gt10', '[Fe/H] > −1', '#e09f3e')]:
    y = q[f'sigma_{key}']
    ax.errorbar(R_MID, y, [y - q[f'sigma_{key}_lo'], q[f'sigma_{key}_hi'] - y], fmt='o', ms=3, lw=1, color=c,
                label=f"giants {label}: $h={summary[key]['h']:.2f}$ kpc")
    m = (R_MID > FIT_RANGE[0]) & (R_MID < FIT_RANGE[1])
    cf = np.polyfit(R_MID[m], np.log(y[m]), 1)
    rr = np.array(FIT_RANGE)
    ax.plot(rr, np.exp(np.polyval(cf, rr)), color=c, lw=.9, ls=(0, (4, 1.5)))
for key, label, c in [('rr_lt15', f'RRab [Fe/H] < {RR_SPLIT:g}', '#3b2c85'), ('rr_gt15', f'RRab [Fe/H] > {RR_SPLIT:g}', '#2a9d8f')]:
    n = rr_n[key]
    ax.errorbar(rr_mid, n / rr_area, np.sqrt(n) / rr_area, fmt='s', ms=3.5, mfc='white', lw=1, color=c,
                label=f"{label}: $h={summary[key]['h']:.2f}$ kpc".replace('-', '−'))
    cf = np.polyfit(rr_mid[fit], np.log(n[fit] / rr_area[fit]), 1, w=np.sqrt(n[fit]))
    ax.plot(RR_FIT_RANGE, np.exp(np.polyval(cf, RR_FIT_RANGE)), color=c, lw=.9, ls=(0, (1, 1)))
ax.set(xlabel='R [kpc]', ylabel=r'number per kpc$^2$', xlim=(0, 10), yscale='log')
ax.set_title('(c) Giants and RR Lyrae by metallicity', pad=62)
ax.legend(loc='lower center', bbox_to_anchor=(.5, 1.02), ncol=1, frameon=False, fontsize=6.5)
fig.savefig(ROOT / 'figures/paper_feh_profile.png', dpi=200)

In [ ]:
from IPython.display import Image
Image(filename='figures/paper_feh_profile.png')

**Figure 5.1.** Metallicity and surface density against in-plane radius $R$ (kpc). (a) Weighted median [Fe/H] of
the BOSS giants ($G\leq$ 17.5, blue filled; $G$ < 16.5, blue open, dashed) and the APOGEE giants
(orange); bars: 16th–84th percentiles of the bootstrap of the median; dashed and dotted lines: published gradients over their radial
ranges, drawn through the BOSS profile at the middle of each range (slopes only); [Choudhury et al. (2016)](https://doi.org/10.1093/mnras/stv2414)
is the middle of their quoted interval. (b) The same for the RRab (green), with the three radial segments of
[Skowron et al. (2016)](https://ui.adsabs.harvard.edu/abs/2016AcA....66..269S) (black dashed). (c) Surface density of the BOSS giants on
the disc plane (number per kpc², logarithmic) in three ranges of true [Fe/H] (filled circles: below -1.5, dark blue;
-1.5 to -1, teal; above -1, yellow), corrected for the BOSS metallicity errors;
dashed: exponentials fitted at 2 < $R$ < 10 kpc, with their scale lengths $h$ in the legend; open
squares: RRab members below and above [Fe/H] = −1.5 (photometric [Fe/H]), Poisson errors, with exponentials fitted at
2 < $R$ < 8 kpc (dotted). Bars: 16th–84th percentiles of 200 bootstraps of
nside = 64 sky pixels and, in (c), of the stars that define the metallicity response. `scripts/paper_feh_profile.py`

Figure 5.1 shows the result. The median [Fe/H] of the BOSS giants falls from -0.54 at
$R$ < 1 kpc to -0.61 at 5–6 kpc and then to
-0.92 at 9–10 kpc. The APOGEE giants follow the same shape but lie
0.05–0.12 dex lower in every bin. The two samples reach different depths on the giant branch:
BOSS goes to $G$ = 17.5, while 93 per cent of the APOGEE weight in its fields is at $G$ < 16.5, because APOGEE's faint
selection groups are mostly uncovered and its weights have no coverage factor. Limited to $G$ < 16.5, the BOSS median
falls by 0.06–0.12 dex inside 6 kpc and agrees with APOGEE to within
0.045 dex out to 8 kpc (open symbols). Part of the excess of the fainter BOSS giants is a
measurement bias: BOSSNet overestimates the metallicity of hot, low-S/N giants, which are common at faint $G$ (chapters 3 and 4), so
the open symbols are the reliable BOSS profile, and we take the gradients from them. Straight lines fitted to the medians of the BOSS
giants with $G$ < 16.5 give -0.008 ± 0.003 dex/kpc inside
6 kpc and -0.068 ± 0.007 dex/kpc beyond, and the APOGEE giants
-0.019 ± 0.002 and -0.077 ± 0.010 dex/kpc; all BOSS giants
to $G$ = 17.5 give -0.015 ± 0.002 and -0.088 ±
0.005 dex/kpc. In every sample the profile is nearly flat inside 6 kpc and falls beyond. The
scale lengths below refer to the BOSS giants with $G\leq$ 17.5; the range above -1 is a total, which
BOSS and APOGEE agree on.

Figure 5.1(c) shows why the median falls: the metal-poor giants are spread more widely than the metal-rich ones. Between
2 and 10 kpc exponentials $\Sigma\propto e^{-R/h}$ fitted to the surface density $\Sigma$ of the
three ranges have scale lengths $h$ = 1.80 (+0.27/−0.11) kpc below
-1.5, 2.20 (+0.31/−0.15) kpc from -1.5 to
-1 and 1.10 (+0.05/−0.04) kpc above -1. The
metal-rich profile is not exponential: it falls by factors of 3.1–4.7 per kpc beyond
6 kpc, against 2.5 for the fit, and this steepening produces the break in the median. Inside
2 kpc the density of metal-poor giants falls below that at 2–3 kpc (85 and
91 against 131 per kpc²); the selection function recovers 0.82 of the expected
metal-poor stars inside 1.5° (chapter 2), and 0.88 ± 0.20 within
3° in the mock, so this dip is at most marginally significant. The foreground matters most for the
metal-poor giants at large radii: set to zero or doubled, it changes $h$ to 2.07 and 1.48
kpc below -1.5, 2.50 and 1.87 kpc from -1.5 to
-1 and 1.19 and 1.03 kpc above -1; a threshold of
S/N ≥ 20 instead of 16 changes them by less than 0.3 kpc (chapter 3), and the
medians by at most 0.035 dex outside 1 kpc.

The RR Lyrae [Fe/H] declines steadily, from -1.32 at the centre to -1.56 at 9–10 kpc
(-0.027 ± 0.002 dex/kpc inside 6 kpc), with a flat stretch between
4 and 7 kpc (medians -1.43 to -1.44). Their surface
density (Figure 5.1c, open squares; to 8 kpc, the largest radius at which their footprint,
$\rho$ < 10° from the centre, covers all azimuths) follows the metal-poor giants between 2 and
8 kpc, where there are 0.83 RRab below [Fe/H] = −1.5 per BOSS giant below −1.5 in every 1-kpc
ring (0.74–0.93, scatter 0.08), although each sample has its own selection
function and [Fe/H] scale: exponentials have $h$ = 2.05 kpc for the RRab below [Fe/H] = −1.5 and
1.64 kpc above, against 1.80 and 2.20 kpc for the giants below −1.5 and from −1.5 to
−1. Inside 2 kpc they part: the RRab below −1.5 rise to 359 per kpc² at $R$ <
1 kpc while the giants below −1.5 dip to 85 (bootstrap 23–229),
so there are 4.2 RRab per metal-poor giant (1.6–15), and
2.0 (1.1–3.2) at 1–2 kpc. These
profiles count stars, not mass: a fixed formed mass of the old metal-poor populations places fewer giants in the selection than the
younger metal-rich ones (an efficiency $\eta\approx$ 1.6 × 10⁻⁴ against
2.4–3.2 × 10⁻⁴ selected giants per M☉ formed for the bins from −2.0 to −1.5 and from −0.75
to −0.25; chapter 4, Figure 4.4a), so in mass the metal-poor profile would lie higher relative to the metal-rich one,
but its scale length would not change if the conversion does not vary with radius.

## Published gradients

The published gradients are drawn in Figure 5.1(a, b) over their own radial ranges, through our profile at the middle
of each range, so that only the slopes are compared; Figure 5.2 compares them range by range. The APOGEE gradient of
[Povick et al. (2025)](https://doi.org/10.1093/mnras/staf1428), -0.0296 ± 0.0017 dex/kpc from
medians in 1-kpc bins out to about 10°, agrees with our APOGEE slope over the same range (-0.028 ±
0.002); the BOSS giants with $G$ < 16.5 give a shallower -0.022 ±
0.002. The slope of [Muñoz et al. (2023)](https://doi.org/10.1051/0004-6361/202347046),
-0.03 dex/deg out to 11°, agrees with both (-0.031 ±
0.002 for APOGEE, -0.031 ± 0.001 for BOSS). The gradients
from photometric metallicities and from the ratio of carbon to M-type stars on the asymptotic giant branch are steeper:
-0.048 ± 0.001 ([Grady, Belokurov & Evans 2021](https://doi.org/10.3847/1538-4357/abd4e4)) and
-0.047 ± 0.003 dex/kpc ([Cioni 2009](https://doi.org/10.1051/0004-6361/200912138)),
1.6–2.8 times our -0.017 to -0.031 dex/kpc over the same
ranges, and -0.049 to -0.066 dex/kpc inside 4 kpc (Choudhury et al.
2016, from the slope of the red giant branch in optical photometry), where ours are -0.004 ±
0.006 (BOSS) and -0.014 ± 0.004 (APOGEE). These methods respond
to the age of the population as well as its metallicity, and [Feast et al. (2010)](https://doi.org/10.1111/j.1745-3933.2010.00933.x)
attributed the outer fall of the carbon-star ratio to older stars rather than to metallicity. The near-infrared giant-branch slopes
of [Choudhury et al. (2021)](https://doi.org/10.1093/mnras/stab2446) give -0.008 ± 0.001
dex/kpc inside 6 kpc, the slope of our BOSS giants with $G$ < 16.5
(-0.008 ± 0.003) and shallower than that of APOGEE (-0.019 ±
0.002). The shape of our profile, flat to about 6 kpc and falling beyond, matches the
fields of [Carrera et al. (2008)](https://doi.org/10.1088/0004-6256/135/3/836), with [Fe/H] from the Ca II triplet (the three strong
lines of ionised calcium near 850 nm) of ≈ -0.5 out to 6° and -0.8 at
8°, and the plateau and drop near 8° in the *Gaia* XP metallicities (from the low-resolution BP/RP spectra of
*Gaia*) of [Massana et al. (2024)](https://doi.org/10.1093/mnras/stad3788); a single linear gradient does not describe it. The RR
Lyrae agree with Skowron et al. (2016) inside 5 kpc (-0.029 ± 0.002 against our
-0.030 ± 0.002 dex/kpc); from 5 to 8 kpc they find no
gradient, and our slope there, -0.023 ± 0.008 dex/kpc, is shallower but not zero.
[Cusano et al. (2021)](https://doi.org/10.1093/mnras/stab901) found no RR Lyrae gradient. From *Gaia* XP metallicities tied to APOGEE,
[Frankel et al. (2025)](https://doi.org/10.3847/1538-4357/ad9b17) found exponential scale lengths that shrink from 2.3 kpc at
[M/H] = −1.8 to 0.75 kpc at [M/H] = −0.25 ([M/H] is the abundance of all metals relative to the Sun's); ours,
1.8–2.2 kpc below [Fe/H] = −1 and 1.1 kpc above, follow the same trend,
with a metal-rich range that spans −1 to above 0 and so mixes their shorter and longer scales.

<details><summary>About <code>scripts/paper_feh_literature.py</code></summary>

```text
Our [Fe/H] profiles against published LMC metallicity gradients (lmcold.constants.LMC_FEH_GRADIENT_LITERATURE, _LEVEL_).

Run scripts/paper_feh_profile.py first. For each published gradient we fit a straight line to our median [Fe/H] in the 1-kpc
in-plane bins whose centres lie inside the published radial range, weighted by the bootstrap errors of the medians (floor 0.005
dex); the error is the formal one of that fit. Giants: the BOSS giants with G < 16.5 ('BOSS G<16.5' of paper_feh_profile.py), the
reliable BOSS profile (fainter, hot, low-S/N giants have overestimated [Fe/H]; Section 3), and APOGEE; our own slopes are also given
for all BOSS giants (G < 17.5) as a check. Ranges with fewer than three bins (Skowron+16 beyond 8 kpc) get no fit.
*: published range given in degrees, converted at 0.873 kpc/deg. (a) giants: BOSS and APOGEE; (b) RRab; (c) median against projected distance
with the mean levels quoted in degrees by Carrera et al. (2008) and Massana et al. (2024; Gaia XP [M/H]) and the
Muñoz et al. (2023) APOGEE slope per degree drawn through our BOSS profile at the middle of its range.
Writes figures/paper_feh_literature.png and results/feh_gradients.csv.

Inputs: results/feh_profile.csv (median [Fe/H] per 1-kpc bin of in-plane radius R, with bootstrap median_lo/median_hi) and
results/feh_profile_rho.csv (the same per 1-deg bin of projected distance rho), both from scripts/paper_feh_profile.py; the
published values in lmcold.constants (LMC_FEH_GRADIENT_LITERATURE [dex/kpc], LMC_FEH_GRADIENT_DEG_LITERATURE [dex/deg],
LMC_FEH_LEVEL_LITERATURE).
Output CSV results/feh_gradients.csv, one row per published gradient and sample, plus our own slopes over 0-6, 6-10 and 0-10 kpc
('this work'): label, sample, r_lo, r_hi (range, kpc or, for the [dex/deg] rows, deg), slope_lit, err_lit (published), slope,
err (ours), and slope_unweighted for the [dex/deg] rows.
Panels (1 x 3; legends above the panels):
(a) x: published giant gradients (labels: study and radial range); y: d[Fe/H]/dR [dex/kpc]; black: published value (square with
    error; x where no error is quoted; a quoted interval of slopes is shown by its midpoint and half-range plus the error);
    coloured circles: our BOSS and APOGEE slopes over the same range.
(b) the same for the RRab (Skowron+16 inner, middle, outer).
(c) x: rho [deg], 0-12; y: median [Fe/H] [dex]; BOSS and APOGEE profiles with bootstrap errors; horizontal segments: Carrera+08
    (mean) and Massana+24 (XP [M/H]) levels over their quoted ranges; purple dashed: the Munoz+23 slope through our BOSS profile.
Placement: appendix E (docs/figures.md). Run: .venv/bin/python scripts/paper_feh_literature.py (after paper_feh_profile.py)
Run time: 1 s on a MacBook (M-series), docs/run_times.txt.
```
</details>

**Data read by `scripts/paper_feh_literature.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `results/feh_profile.csv` | repository | written by `python scripts/paper_feh_profile.py` |
| `results/feh_profile_rho.csv` | repository | written by `python scripts/paper_feh_profile.py` |

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lmcold.constants import LMC_FEH_GRADIENT_DEG_LITERATURE, LMC_FEH_GRADIENT_LITERATURE, LMC_FEH_LEVEL_LITERATURE
from lmcold.layout import panel_grid
from lmcold.sources import ROOT

### Load the data
Our [Fe/H] profiles written by `scripts/paper_feh_profile.py`: `results/feh_profile.csv` (against in-plane radius) and
`results/feh_profile_rho.csv` (against projected distance). The published gradients and levels are not read from a file: they
are listed in `lmcold.constants`.

In [ ]:
# results/feh_profile.csv: median [Fe/H] and its bootstrap range per sample and bin of in-plane radius R [kpc]
t = pd.read_csv(ROOT / 'results/feh_profile.csv')
# results/feh_profile_rho.csv: the same per bin of projected distance rho [deg]
p = pd.read_csv(ROOT / 'results/feh_profile_rho.csv')

### Our profiles
Median [Fe/H] per bin of in-plane radius and of projected distance, with the error of each median.

In [ ]:
BRIGHT = 'BOSS G<16.5'  # sample name in results/feh_profile*.csv (paper_feh_profile.G_BRIGHT)
GIANTS = [BRIGHT, 'APOGEE']  # giant samples compared with the published giant gradients
t['R'] = .5 * (t.R_lo + t.R_hi)  # bin centre [kpc]
t['e'] = (.5 * (t.median_hi - t.median_lo)).clip(.005)  # error of the median: half the 16-84 bootstrap range, floor 0.005 dex
p['x'], p['e'] = .5 * (p.rho_lo + p.rho_hi), (.5 * (p.median_hi - p.median_lo)).clip(.005)  # bin centre [deg], error [dex]


def slope(sample, lo, hi):
    """Weighted straight-line fit of our median [Fe/H] against R over the bins with centres in (lo, hi).

    Parameters
    ----------
    sample : str
        'BOSS', 'APOGEE' or 'RRab' (rows of results/feh_profile.csv).
    lo, hi : float
        Radial range [kpc]; bin centres strictly inside are used.

    Returns
    -------
    slope, err : float
        Gradient [dex/kpc] and its formal error (polyfit weights 1 / e, unscaled covariance); NaN for fewer than three bins.
    """
    q = t[(t['sample'] == sample) & (t.R > lo) & (t.R < hi)]
    if len(q) < 3:  # a line through two bins has no meaningful error
        return np.nan, np.nan
    # Line [Fe/H] = p[0] R + p[1]; p[0] is the slope [dex/kpc].
    p, cov = np.polyfit(q.R, q['median'], 1, w=1 / q.e, cov='unscaled')
    return p[0], np.sqrt(cov[0, 0])

### Our slopes over each published range
Written to `results/feh_gradients.csv`, with our own inner and outer slopes.

In [ ]:
lit = pd.DataFrame(LMC_FEH_GRADIENT_LITERATURE)  # one row per published gradient [dex/kpc]: label, tracer, r_lo, r_hi, slope, err
rows = []  # rows of results/feh_gradients.csv
# Our slope over each published range, for the giant samples (giant gradients) or the RRab (RR Lyrae gradients).
for _, L in lit.iterrows():
    for sample in (GIANTS if L.tracer == 'giants' else ['RRab']):
        rows.append(dict(label=L.label, sample=sample, r_lo=L.r_lo, r_hi=L.r_hi, slope_lit=L.slope, err_lit=L.err,
                         **dict(zip(['slope', 'err'], slope(sample, L.r_lo, L.r_hi)))))
for sample in GIANTS + ['BOSS', 'RRab']:  # our own inner and outer slopes, either side of the break near 6 kpc (all BOSS as a check)
    for lo, hi in [(0., 6.), (6., 10.), (0., 10.)]:
        rows.append(dict(label='this work', sample=sample, r_lo=lo, r_hi=hi, **dict(zip(['slope', 'err'], slope(sample, lo, hi)))))
for L in LMC_FEH_GRADIENT_DEG_LITERATURE:  # [dex/deg], against projected distance (no minimum number of bins)
    for sample in GIANTS:
        q = p[(p['sample'] == sample) & (p.x > L['rho_lo']) & (p.x < L['rho_hi'])]
        # Weighted slope [dex/deg] with formal error; the unweighted slope is stored too.
        c, cov = np.polyfit(q.x, q['median'], 1, w=1 / q.e, cov='unscaled')
        rows.append(dict(label=L['label'] + ' [dex/deg]', sample=sample, r_lo=L['rho_lo'], r_hi=L['rho_hi'], slope_lit=L['slope'],
                         slope=c[0], err=np.sqrt(cov[0, 0]), slope_unweighted=np.polyfit(q.x, q['median'], 1)[0]))
g = pd.DataFrame(rows)
g.to_csv(ROOT / 'results/feh_gradients.csv', index=False)
print(g.round(4).to_string(index=False))

### The figure

In [ ]:
plt.style.use(ROOT / 'src/lmcold/paper.mplstyle')
colours = {BRIGHT: '#2878b5', 'APOGEE': '#d75a20', 'RRab': '#3a923a'}  # one colour per sample
# Three equal panels (sizes in inches); the large bottom margin holds the rotated tick labels of (a) and (b).
fig, axes = panel_grid(1, 3, width=7, left=.55, right=.1, hgap=.55, bottom=1.05, top=.6)
for ax, tracer, samples in [(axes[0, 0], 'giants', GIANTS), (axes[0, 1], 'RRab', ['RRab'])]:
    L = lit[lit.tracer == tracer].reset_index(drop=True)  # the published gradients of this tracer; x position = row index
    for i, r in L.iterrows():
        # A quoted interval of slopes (slope to slope_hi) is drawn at its midpoint with half its range added to the error.
        lo, hi = sorted([r.slope, r.slope if pd.isna(r.get('slope_hi')) else r.slope_hi])
        e = 0. if r.err is None or pd.isna(r.err) else r.err
        ax.errorbar(i - .15, .5 * (lo + hi), [[.5 * (hi - lo) + e]] * 2, fmt='s' if e else 'x', ms=4, lw=1, color='k',
                    label='published' if i == 0 else None)
        # Our slopes over the same range, offset to the right of the published value, one per sample.
        for j, s in enumerate(samples):
            q = g[(g.label == r.label) & (g['sample'] == s)].iloc[0]
            ax.errorbar(i + .1 + .15 * j, q.slope, q.err, fmt='o', ms=4, lw=1, color=colours[s],
                        label=f"this work, {s.replace('<', ' < ')}" if i == 0 else None)
    ax.axhline(0, color='0.7', lw=.6, zorder=0)
    # Tick labels: study (without the Skowron+16 range names) and radial range; * marks ranges converted from degrees.
    ax.set_xticks(range(len(L)))
    ax.set_xticklabels([f"{r.label.replace(' inner', '').replace(' middle', '').replace(' outer', '')}\n"
                        f"{r.r_lo:g}–{r.r_hi:g} kpc{'*' if r.range_from_deg else ''}" for _, r in L.iterrows()],
                       rotation=60, ha='right', fontsize=6.5)
    ax.set(xlim=(-.6, len(L) - .4), ylabel=r'd[Fe/H]/dR [dex kpc$^{-1}$]')
    ax.legend(loc='lower center', bbox_to_anchor=(.5, 1.08), ncol=1, frameon=False, fontsize=6.5)
axes[0, 0].set_title('(a) Giants', pad=46)
axes[0, 1].set_title('(b) RR Lyrae', pad=46)
# (c) Our median [Fe/H] against projected distance, with bootstrap errors.
ax = axes[0, 2]
for s in GIANTS:
    q = p[p['sample'] == s]
    ax.errorbar(.5 * (q.rho_lo + q.rho_hi), q['median'], [q['median'] - q.median_lo, q.median_hi - q['median']], fmt='-o', ms=3,
                lw=1, color=colours[s], label=f"this work, {s.replace('<', ' < ')} median")
# Published mean levels as horizontal segments over their quoted rho ranges; one legend entry per study.
for name, style in [('Carrera+08', dict(color='k', lw=2.2)), ('Massana+24', dict(color='0.55', lw=2.2, ls=(0, (3, 1))))]:
    for k, L in enumerate(x for x in LMC_FEH_LEVEL_LITERATURE if x['label'] == name):
        ax.plot([L['rho_lo'], L['rho_hi']], [L['feh']] * 2, **style, label=name + (' (XP [M/H])' if 'Massana' in name else ' (mean)')
                if k == 0 else None)
bq = p[p['sample'] == BRIGHT]  # our bright-BOSS profile against rho
for L in LMC_FEH_GRADIENT_DEG_LITERATURE:  # through our BOSS profile at the middle of the range: slope only
    rr = np.array([L['rho_lo'], L['rho_hi']])  # the published range [deg]
    # Our BOSS median at the middle of the range (linear interpolation between bin centres) anchors the published slope.
    y0 = np.interp(rr.mean(), .5 * (bq.rho_lo + bq.rho_hi), bq['median'])
    ax.plot(rr, y0 + L['slope'] * (rr - rr.mean()), color='#7b4fa0', lw=1.1, ls=(0, (4, 1.5)),
            label=f"{L['label']} ({L['slope']:g} dex/deg)")
# Axes, title and the legend above the panel.
ax.set(xlabel=r'$\rho$ [deg]', ylabel='[Fe/H]', xlim=(0, 12))
ax.set_title('(c) Against projected distance', pad=46)
ax.legend(loc='lower center', bbox_to_anchor=(.5, 1.08), ncol=2, frameon=False, fontsize=6.5, columnspacing=.8, handlelength=1.6)
fig.savefig(ROOT / 'figures/paper_feh_literature.png', dpi=200)

In [ ]:
from IPython.display import Image
Image(filename='figures/paper_feh_literature.png')

**Figure 5.2.** Our metallicity gradients against published ones. (a) Giants: radial gradient d[Fe/H]/d$R$
(dex/kpc) of each published study (black squares, labelled with study and radial range; the bar of Choudhury et al. 2016 spans their
quoted interval and errors) and ours over the same in-plane range from BOSS with $G$ < 16.5 (blue) and APOGEE
(orange); * marks ranges converted from degrees. (b) The same for the RRab (green) and the three segments of Skowron et al. (2016);
the cross marks their 5–8 kpc segment, for which they quote no gradient and no error. (c)
Median [Fe/H] against projected distance $\rho$ (deg) for BOSS with $G$ < 16.5 (blue) and APOGEE (orange), with the
mean levels quoted by Carrera et al. (2008; black bars) and Massana et al. (2024; grey dashed bars) and the slope of Muñoz et al.
(2023) drawn through the BOSS profile at the middle of its range (purple dashed). `scripts/paper_feh_literature.py`

### How the published gradients are compared

For every study we fit a straight line to our medians in the 1-kpc bins whose centres lie inside its range, weighted by their
bootstrap errors (floor 0.005 dex), and quote the formal error of the fit; ranges with fewer than
3 of our bins, as beyond 8 kpc for the RR Lyrae, are not fitted. Where a study gives its
extent in degrees we convert at 0.874 kpc per degree ($D_0\,\pi/180$), so those ranges are approximate.
Because our profile is curved, the fitted slope depends on the range and on the weighting: unweighted, the slope of the BOSS medians
($G$ < 16.5) out to 11° is -0.033 rather than -0.031
dex/deg. For the giants we use the BOSS giants with $G$ < 16.5, the reliable BOSS profile, and APOGEE. The published
values are from the abstracts and were checked against the published full texts; the preprint of Povick et al. (2025) quotes
−0.0380 ± 0.0022 dex/kpc, the published version -0.0296 ± 0.0017, which we use. Panel (c) of
Figure 5.2 shows the levels that Carrera et al. (2008) and Massana et al. (2024) quote against projected distance;
they are on their own metallicity scales (Ca II triplet mean [Fe/H]; *Gaia* XP [M/H]), so only the shape compares with ours. The RR
Lyrae metallicities of Skowron et al. (2016) are on the high-dispersion scale of
[Jurcsik (1995)](https://ui.adsabs.harvard.edu/abs/1995AcA....45..653J) and ours on that of Crestani et al. (2021), which is the scale of
[Carretta et al. (2009)](https://doi.org/10.1051/0004-6361/200913003) plus 0.08 dex. Skowron et al. (2016)
themselves convert to the Carretta scale with [Fe/H]<sub>C09</sub> = 1.001 [Fe/H]<sub>J95</sub> − 0.112, a slope of unity, so their
gradients apply unchanged on our scale; their alternative route through the scale of [Zinn & West (1984)](https://doi.org/10.1086/190947),
[Fe/H]<sub>ZW</sub> = ([Fe/H]<sub>J95</sub> − 0.88)/1.431 followed by [Fe/H]<sub>C09</sub> = 1.105 [Fe/H]<sub>ZW</sub> + 0.160, has a
slope of 0.77 and would make their inner gradient -0.022 dex/kpc. Two further differences limit the
comparison: their radius is the three-dimensional distance from the centre, with each star's own photometric distance, rather than
our in-plane $R$; and with a different light-curve relation their whole-galaxy gradient falls from −0.019 to −0.011 dex/kpc
(Skowron et al. 2016).

## Published metallicity distributions

Figure 5.3 compares, by number of giants, our [Fe/H] distributions with published ones in the same places. Ours are
the BOSS giants with their calibrated [Fe/H] as measured (the response correction of chapter 4 acts on binned counts, not on stars)
and the APOGEE giants, each with its selection weights. In the bar we take our giants within 0.5° of each of the
7 fields of [Cole et al. (2005)](https://doi.org/10.1086/428007), and in the disc those within
0.5° of the fields of Carrera et al. (2008) at 5, 6 and 8° from the centre (their 3° field holds 4 BOSS giants), reweighted so
that each field has its share of the published sample. For Carrera et al. (2008) we compute [Fe/H] from their Ca II triplet index with
their equation (1), on the scale of [Carretta & Gratton (1997)](https://doi.org/10.1051/aas:1997116), for their radial-velocity
members.

The Ca II triplet distributions are more metal-rich than ours: in the bar the median is -0.40 (Cole et al. 2005)
against -0.53 (BOSS) and -0.60 (APOGEE), and 73 per cent of the stars lie above
−0.5; in the disc fields it is -0.59 (Carrera et al. 2008) against -0.63 and -0.69,
with a broader metal-poor tail (26 per cent below −1, against 14 and
13). Much of the offset is the metallicity scale. For the stars that are in both samples, BOSS minus Ca II
triplet [Fe/H] is -0.18 (79 stars of Carrera et al. 2008, robust dispersion 0.33 dex)
and APOGEE minus Ca II triplet -0.26 (24 stars, 0.18 dex); only
3 stars of Cole et al. (2005) are in BOSS (-0.21).

Over the whole LMC the selection weights move the APOGEE median from -0.73 as targeted to -0.67, the
share above −0.5 from 7 to 13 per cent and the share below −1 from
19 to 13 per cent (Figure 5.3c). The BOSS giants as measured have a median
of -0.59 and 33 per cent above −0.5, which includes the overestimated metallicities of hot,
low-S/N giants (chapter 3). Panel (d) shows the 31 545 BOSS giants that also have the *Gaia* XP Strömgren metallicities of
[Omkumar et al. (2025)](https://doi.org/10.1051/0004-6361/202452510): their median, -0.61, matches the BOSS one,
-0.64, but their distribution is much broader (41 per cent above −0.5 and 22
per cent below −1, against 29 and 17), while the XP [M/H] of
[Andrae, Rix & Chandra (2023)](https://doi.org/10.3847/1538-4365/acd53e) for the same stars, converted to [Fe/H], peaks about
0.2 dex lower (median -0.83).

<details><summary>About <code>scripts/paper_mdf_literature.py</code></summary>

```text
Paper figure (appendix E): metallicity distributions of LMC giants by number, ours against published ones in the same places.

Ours: BOSS giants (calibrated [Fe/H], weights w x coverage x (1 - f_MW)) and APOGEE giants (weights w) of lmcold.mass_samples, as
measured (no response correction, which acts on binned counts, not on stars). Published (scripts/fetch_mdf_literature.py):
(a) bar: Cole et al. (2005) Ca II triplet [Fe/H]; ours within FIELD_R of the centre of each of their seven fields.
(b) disc fields 5, 6 and 8 deg north of the bar: Carrera et al. (2008) Ca II triplet, [Fe/H] from their equation (1) on the CG97
    scale, -2.95 + 0.38 Sum(Ca) + 0.23 M_I, M_I = I - LMC_DISTANCE_MODULUS - A_I (A_I = 1.505 E(B-V), lmcold.extinction), radial
    velocity members (170-380 km/s, their criterion); ours within FIELD_R of each field. Their 3-deg field holds 4 BOSS giants and is
    left out. In (a) and (b) our stars are reweighted so that each field has its share of the published sample.
(c) whole LMC within 12 deg: APOGEE as targeted (unweighted) and with its selection weights; BOSS weighted.
(d) the same BOSS giants with Gaia XP photometric metallicities: Omkumar et al. (2025) Stroemgren [Fe/H] and Andrae et al. (2023)
    [M/H] (to [Fe/H] with the alpha trend, as in paper_mass.py), BOSS weights; BOSS spectroscopic [Fe/H] of the same stars.
Scale check: stars of Cole+05 and Carrera+08 also in BOSS or APOGEE (within MATCH arcsec), median and robust sd of survey minus
Ca II triplet [Fe/H]. Histograms per dex of [Fe/H], unit area; medians in the legends. Writes results/mdf_literature.json and figures/paper_mdf_literature.png.

Inputs: lmcold.mass_samples.samples() (BOSS and APOGEE mass samples; BOSS also carries its XP [M/H], mh_xp);
data/giant_foreground.parquet (scripts/build_giant_mass.py: coverage fill and foreground fraction f_mw per BOSS giant);
results/mass_alpha_trend.csv (scripts/build_giant_mass.py); sources.yaml mdf_cole05, mdf_carrera08 (columns used: ra, dec,
sum_ca, Imag, rv) and mdf_omkumar25 (source_id, feh). SFD E(B-V) for the Carrera+08 stars comes from the `dustmaps` package
(SFDQuery); its map files are the data/sources.yaml entry sfd_dust_map, downloaded on first use.
Panels (2 x 2; x: [Fe/H] [dex], -2.5 to 0.6; y: fraction per dex, histograms in 0.1-dex bins normalised to unit area over the
plotted range): (a) Cole+05 bar fields with BOSS and APOGEE in the same fields; (b) Carrera+08 fields at 5, 6, 8 deg with
BOSS and APOGEE; (c) APOGEE unweighted and weighted, BOSS weighted, rho < 12 deg; (d) BOSS giants with XP [M/H]: Omkumar+25,
Andrae+23 (converted) and BOSS spectroscopic [Fe/H], all with the BOSS weights. Legends (above each panel) give the number of
stars and the weighted median.
Output JSON: scale_check (n, median and robust sd of survey minus literature [Fe/H] for matched stars) and, per panel and curve,
n, weighted median, and weighted fractions above -0.5 and below -1. Paper appendix E (Sect. E2; docs/figures.md).
Run: .venv/bin/python scripts/paper_mdf_literature.py
Run time: 2 s on a MacBook (M-series), docs/run_times.txt.
```
</details>

**Data read by `scripts/paper_mdf_literature.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `data/apogee_weights.parquet` | Zenodo bundle 1 | APOGEE selection weights per candidate star (selection groups, coverage, w = N_cand / N_quality x pass factor in H x XP [M/H]) |
| `data/boss_weights.parquet` | Zenodo bundle 1 | BOSS selection weights per eligible parent star (selection groups, w = N_elig / N_good x pass factor in G x XP [M/H] x ring) |
| `data/giant_foreground.parquet` | Zenodo bundle 1 | Milky Way foreground model per giant, from the mass analysis |
| `data/mdf_carrera08.csv` | public archive | Ca II triplet index, V, I and radial velocity of stars in four LMC fields 3-8 deg north of the bar (Carrera et al. 2008, 2008AJ....135..836C; 635 rows in VizieR) |
| `data/mdf_cole05.csv` | public archive | Ca II triplet [Fe/H] (CG97 scale) of 373 LMC bar red giants (Cole et al. 2005, 2005AJ....129.1465C) |
| `data/mdf_omkumar25.parquet` | public archive | Stroemgren photometric [Fe/H] from Gaia DR3 XP spectra of LMC red giants (Omkumar et al. 2025, 2025A&A...700A..74O, table lmc-rgb) |
| `data/raw/skowron21/table1.dat.gz` | public archive | OGLE-IV red-clump E(V-I) map of the Magellanic Clouds, 81,684 boxes of 3.4-27.5 arcmin; Skowron et al. 2021, ApJS 252, 23 (2021ApJS..252...23S, doi:10.3847/1538-4365/abcb81) |
| `data/spec_giants.parquet` | Zenodo bundle 1 | APOGEE DR17 and SDSS-V DR20 BOSS spectra of LMC giant candidates before the Gaia match |
| `results/boss_calibration.json` | repository | calibration of the BOSS [Fe/H] and velocities to APOGEE on the stars observed by both; written by `python scripts/build_giants.py` |
| `results/mass_alpha_trend.csv` | repository | written by `python scripts/build_giant_mass.py` |

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

In [ ]:
import json
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import astropy.units as u
from astropy.coordinates import SkyCoord
from dustmaps.sfd import SFDQuery
from dustmaps.std_paths import data_dir

from lmcold.constants import LMC, LMC_DISTANCE_MODULUS
from lmcold.extinction import rc_map, true_ebv
from lmcold.layout import panel_grid
from lmcold.maps import wquantile
from lmcold.mass_samples import samples
from lmcold.sources import DATA, ROOT

### Settings

In [ ]:
FIELD_R = .5  # deg, radius about each literature field centre within which our stars are taken
MATCH = 1.5  # arcsec, positional match radius for the scale check
EDGES = np.arange(-2.5, .61, .1)  # [Fe/H] histogram edges, 0.1 dex

### Load the data
The BOSS and APOGEE mass samples through `lmcold.mass_samples.samples()` (built from the catalogues of Zenodo bundle 1), the
coverage fill and foreground fraction of each BOSS giant (`data/giant_foreground.parquet`, written by
`scripts/build_giant_mass.py`), the published Ca II triplet samples of Cole et al. (2005) and Carrera et al. (2008)
(`data/mdf_cole05.csv`, `data/mdf_carrera08.csv`) and the APOGEE [alpha/Fe] trend `results/mass_alpha_trend.csv`. Read where
they are used: the SFD and Skowron et al. (2021) reddening maps (`dustmaps`, `lmcold.extinction.rc_map()`) for the Carrera
stars, and the Omkumar et al. (2025) XP [Fe/H] (`data/mdf_omkumar25.parquet`), joined to the BOSS giants on Gaia source_id.

In [ ]:
# lmcold.mass_samples.samples(): BOSS and APOGEE giants with calibrated [Fe/H], XP [M/H] and selection weight w
b, a, _ = samples()
# data/giant_foreground.parquet: per BOSS giant the coverage fill and the Milky Way foreground fraction f_mw
fg = pd.read_parquet(DATA / 'giant_foreground.parquet')
# data/mdf_cole05.csv: Cole et al. (2005) bar giants with [Fe/H]; field = name before '/'
cole = pd.read_csv(DATA / 'mdf_cole05.csv').assign(field=lambda t: t.Field.str.split('/').str[0])  # field name before '/'
# data/mdf_carrera08.csv: Carrera et al. (2008) disc giants (Ca II triplet sum_ca, I magnitude, rv), velocity members kept
car = pd.read_csv(DATA / 'mdf_carrera08.csv').query('170 <= rv <= 380')  # radial-velocity members [km/s], their criterion
# results/mass_alpha_trend.csv: the APOGEE [alpha/Fe] trend against [Fe/H]
trend = pd.read_csv(ROOT / 'results/mass_alpha_trend.csv')
# The SFD map (data/sources.yaml sfd_dust_map, two 64-MB files): downloaded into the dustmaps data folder if it is not there yet,
# from the copy at github.com/kbarbary/sfddata (dustmaps.sfd.fetch() fails: the Harvard Dataverse answers 403); both files have the
# sha256 recorded in sources.yaml (checked 2026-11-24)
for name in ('SFD_dust_4096_ngp.fits', 'SFD_dust_4096_sgp.fits'):
    sfd_file = Path(data_dir()) / 'sfd' / name
    if not sfd_file.exists():
        sfd_file.parent.mkdir(parents=True, exist_ok=True)
        urlretrieve(f'https://github.com/kbarbary/sfddata/raw/master/{name}', sfd_file)

### Our giants
BOSS and APOGEE giants with their weights.

In [ ]:
# BOSS weight: selection weight x coverage fill x (1 - Milky Way foreground fraction); APOGEE: selection weight.
b = b.merge(fg[['source_id', 'fill', 'f_mw']], on='source_id', validate='one_to_one').assign(W=lambda t: t.w * t['fill'] * (1 - t.f_mw))
a = a.assign(W=a.w)
centre = SkyCoord(LMC['ra0'], LMC['dec0'], unit='deg')

### The published samples
The bar fields of Cole et al. (2005) and the disc fields of Carrera et al. (2008), with [Fe/H] from their equation (1).

In [ ]:
pos = SkyCoord(car.ra, car.dec, unit='deg')
ebv = true_ebv(car.ra.to_numpy(), car.dec.to_numpy(), SFDQuery()(pos), rc_map())  # Skowron+21 map inside it, SFD outside
# Carrera+08 Eq. (1) with M_I from the photometric LMC distance modulus; field: 1, 2, 3 for projected distance 4-5.5, 5.5-7 and
# > 7 deg (their 5, 6 and 8 deg fields), 0 for the 3-deg field (< 4 deg), which is dropped.
car = car.assign(feh=-2.95 + .38 * car.sum_ca + .23 * (car.Imag - LMC_DISTANCE_MODULUS - 1.505 * ebv),
                 field=np.digitize(pos.separation(centre).deg, [4, 5.5, 7]))
car = car[car.field > 0]


def in_fields(lit, t):
    """Stars of t within FIELD_R of the centre of each field of lit, reweighted to the share of lit in that field.

    Parameters
    ----------
    lit : pandas.DataFrame
        Literature stars with ra, dec [deg] and `field`; a field's centre is the mean ra, dec of its stars.
    t : pandas.DataFrame
        Our sample, with ra, dec [deg] and weight W.

    Returns
    -------
    pandas.DataFrame
        Rows of t in the fields (a star near two field centres appears twice), with W rescaled so that the total weight in each
        field equals that field's fraction of the literature stars (total 1).
    """
    parts = []
    for f, d in lit.groupby('field'):
        # Separation [deg] of each of our stars from the field centre (mean position of the field's literature stars).
        sep = SkyCoord(t.ra, t.dec, unit='deg').separation(SkyCoord(d.ra.mean(), d.dec.mean(), unit='deg')).deg
        s = t[sep < FIELD_R]
        # Our weights in the field normalised to sum to the field's share of the literature sample.
        parts.append(s.assign(W=s.W / s.W.sum() * len(d) / len(lit)))
    return pd.concat(parts)

### Gaia XP metallicities of the same BOSS giants

In [ ]:
# Panel (d): BOSS giants with XP [M/H] (Andrae+23) and with Omkumar+25 [Fe/H] (inner join on Gaia source_id).
xp = b.dropna(subset=['mh_xp'])
# [Fe/H] = [M/H] - log10(0.638 x 10^[alpha/Fe] + 0.362) (Salaris1993 in refs.bib), alpha trend interpolated at [M/H].
xp = xp.assign(feh_xp=xp.mh_xp - np.log10(.638 * 10 ** np.interp(xp.mh_xp, trend.feh, trend.alpha) + .362))
xp = xp.merge(pd.read_parquet(DATA / 'mdf_omkumar25.parquet', columns=['source_id', 'feh']).rename(columns={'feh': 'feh_omk'}),
              on='source_id', validate='one_to_one')
# Line colours of the curves; one() gives unit weights to the unweighted samples.
blue, orange, purple, green = '#2878b5', '#d75a20', '#7b3294', '#1b9e77'
one = lambda t: np.ones(len(t))  # noqa: E731
# Per panel: title and curves (values, weights, colour, line style, label).
panels = [
    ('(a) Bar', [(cole.feh, one(cole), 'k', '-', 'Cole+05, Ca II triplet'),
                 (in_fields(cole, b).feh, in_fields(cole, b).W, blue, '-', 'BOSS, as measured'),
                 (in_fields(cole, a).feh, in_fields(cole, a).W, orange, '--', 'APOGEE')]),
    (r'(b) Fields 5, 6, 8$^\circ$ north', [(car.feh, one(car), 'k', '-', 'Carrera+08, Ca II triplet'),
                                       (in_fields(car, b).feh, in_fields(car, b).W, blue, '-', 'BOSS, as measured'),
                                       (in_fields(car, a).feh, in_fields(car, a).W, orange, '--', 'APOGEE')]),
    (r'(c) $\rho<12^\circ$, spectroscopic', [(a.feh, one(a), orange, ':', 'APOGEE, as targeted'),
                                            (a.feh, a.W, orange, '--', 'APOGEE, weighted'), (b.feh, b.W, blue, '-', 'BOSS, weighted, as measured')]),
    ('(d) Same BOSS giants, Gaia XP', [(xp.feh_omk, xp.W, green, '-', 'Omkumar+25, XP Strömgren'),
                                      (xp.feh_xp, xp.W, purple, '--', 'Andrae+23, XP'), (xp.feh, xp.W, blue, '-', 'BOSS, as measured')]),
]

### Scale check
Stars in both a published sample and ours: survey minus Ca II triplet [Fe/H].

In [ ]:
# Scale check: each literature star matched to its nearest survey star; pairs within MATCH arcsec give survey - literature [Fe/H].
out = {'scale_check': {}}
for lit_name, lit in [('Cole+05', cole), ('Carrera+08', car)]:
    for name, t in [('BOSS', b), ('APOGEE', a)]:
        # i: index of the nearest survey star; d: its separation (angle).
        i, d, _ = SkyCoord(lit.ra, lit.dec, unit='deg').match_to_catalog_sky(SkyCoord(t.ra, t.dec, unit='deg'))
        m = d < MATCH * u.arcsec
        diff = t.feh.to_numpy()[i[m]] - lit.feh.to_numpy()[m]
        out['scale_check'][f'{name} - {lit_name}'] = dict(n=int(m.sum()), median=float(np.median(diff)) if m.any() else None,
                                                         robust_sd=float(1.4826 * np.median(np.abs(diff - np.median(diff)))) if m.any() else None)

### The figure
Histograms per panel; medians and fractions written to `results/mdf_literature.json`.

In [ ]:
plt.style.use(ROOT / 'src/lmcold/paper.mplstyle')
# 2 x 2 equal panels (sizes in inches); the large vertical gap holds the legends above the lower panels.
fig, axes = panel_grid(2, 2, width=7, left=.55, right=.1, hgap=.6, vgap=1.25, bottom=.5, top=.9, aspect=.75)
for ax, (title, curves) in zip(axes.flat, panels):
    out[title] = {}  # JSON entries of this panel, one per curve
    for x, w, colour, ls, lab in curves:
        x, w = np.asarray(x), np.asarray(w)  # [Fe/H] [dex] and weights of one curve
        h = np.histogram(x, EDGES, weights=w)[0]  # weighted counts per 0.1-dex bin
        med = wquantile(x, w, .5)  # weighted median [Fe/H]
        # Number of stars, weighted median, weighted fractions with [Fe/H] > -0.5 and < -1 (over all stars, not only EDGES).
        out[title][lab] = dict(n=len(x), median=float(med), frac_above_m05=float(w[x > -.5].sum() / w.sum()),
                               frac_below_m1=float(w[x < -1].sum() / w.sum()))
        # Histogram normalised to unit area over the plotted range: fraction per bin divided by the 0.1-dex bin width.
        ax.stairs(h / h.sum() / .1, EDGES, color=colour, ls=ls, lw=1.3, label=f'{lab} ({len(x):,}; median {med:.2f})')
    # Axes, panel title, and the legend above the panel.
    ax.set(xlim=(-2.5, .6), ylim=(0, None), xlabel='[Fe/H]', ylabel='fraction per dex', title=title)
    ax.legend(loc='lower center', bbox_to_anchor=(.5, 1.12), frameon=False, fontsize=6.5)
# Write the numbers, print them, save the figure.
(ROOT / 'results/mdf_literature.json').write_text(json.dumps(out, indent=1))
print(json.dumps(out, indent=1))
fig.savefig(ROOT / 'figures/paper_mdf_literature.png', dpi=200)

In [ ]:
from IPython.display import Image
Image(filename='figures/paper_mdf_literature.png')

**Figure 5.3.** [Fe/H] distributions of LMC giants by number: fraction per dex against [Fe/H] (dex), histograms
normalised to unit area, with numbers of stars and medians in the legends above each panel. (a) Bar: Ca II triplet [Fe/H] of Cole et
al. (2005) (black) and our BOSS (blue; calibrated [Fe/H] as measured) and APOGEE (orange dashed) giants within 0.5° of
their fields. (b) The fields of Carrera et al. (2008) at 5, 6 and 8° north of the bar, [Fe/H] from their equation (1), with BOSS and
APOGEE in the same fields. (c) All giants within 12°: APOGEE as targeted (dotted) and with selection weights (dashed),
BOSS with weights (blue). (d) The same BOSS giants with *Gaia* XP metallicities: Strömgren [Fe/H] of Omkumar et al. (2025) (green),
[M/H] of Andrae et al. (2023) converted to [Fe/H] (purple dashed) and BOSS (blue). The Ca II triplet scale lies
0.2 dex above the BOSS and APOGEE scales for stars in common. `scripts/paper_mdf_literature.py`